# Investment Banking Vault — Transparent Initial Creation (“Day 0”)

This notebook reconstructs the **original starting vault** from nothing:

1. declare the fictional company universe and deterministic seed;
2. generate exactly 100 synthetic companies across ten sectors;
3. calculate operating, valuation and qualitative fields;
4. create Obsidian-compatible Markdown, YAML frontmatter, wikilinks, Bases and JSON Canvas files;
5. create the master CSV and data dictionary;
6. validate company identities, enterprise-value equations, wikilinks and Canvas syntax;
7. preview the staged vault and its exact file manifest;
8. copy it to Google Drive only after an explicit human commit.

This is the notebook for the initial creation—not Manual Loop 001 or 002. It intentionally contains **no SYN-101, SYN-102, environment scenario, opportunity score or automated research**.

> Every company and number is fictional. The output is for investment-banking workflow design and education, not investment advice or a real mandate.


## Safety architecture

The notebook always builds inside temporary staging first. Staging may be recreated safely when a cell is rerun. Google Drive is treated separately:

- `COMMIT_TO_DRIVE = False` means nothing is copied into Drive.
- The commit cell refuses to overwrite an existing vault.
- If the target name already exists, choose a new name or move the existing vault yourself after reviewing it.

This recreates the behavior of the first day, when the target vault did not yet exist.


In [ ]:
# 1. Imports and explicit configuration

from pathlib import Path
from collections import Counter
from html import escape
import csv, hashlib, json, math, os, random, re, shutil, tempfile

from IPython.display import display, HTML, Markdown

SEED = 20260717
TARGET_VAULT_NAME = "Alejandro-Reynoso-Investment-Banking-Vault"

# SAFETY GATE: leave False for the complete first run.
COMMIT_TO_DRIVE = False

print("Deterministic seed:", SEED)
print("Target vault:", TARGET_VAULT_NAME)
print("COMMIT_TO_DRIVE:", COMMIT_TO_DRIVE)


In [ ]:
# 2. Mount Google Drive and establish separate staging and target locations

import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    MY_DRIVE = Path("/content/drive/MyDrive")
    WORK_ROOT = Path("/content/investment_banking_vault_day0_build")
else:
    MY_DRIVE = Path.cwd()
    WORK_ROOT = Path(tempfile.gettempdir()) / "investment_banking_vault_day0_build"

STAGING_ROOT = WORK_ROOT / TARGET_VAULT_NAME
TARGET_ROOT = MY_DRIVE / TARGET_VAULT_NAME
STAGING_ZIP = WORK_ROOT / f"{TARGET_VAULT_NAME}-Day-0.zip"

print("Temporary staging:", STAGING_ROOT)
print("Drive destination:", TARGET_ROOT)
print("Destination already exists:", TARGET_ROOT.exists())


## 3. Synthetic-universe specification and vault writer

The following cell is deliberately long because the notebook is self-contained. It records:

- all ten sectors, company names, subsectors, products, customers and revenue models;
- lifecycle, growth, margin, capital-structure and transaction-angle logic;
- the complete 100-company schema;
- Markdown/YAML rendering;
- sector, investment-profile and thematic indices;
- Obsidian Bases and JSON Canvas generation.

No external dataset or hidden model call is involved.


In [ ]:
from __future__ import annotations

import csv
import json
import math
import random
import re
import shutil
from collections import Counter, defaultdict
from pathlib import Path


SEED = 20260717
random.seed(SEED)

ROOT = Path("Alejandro-Reynoso-Investment-Banking-Vault")  # Overridden with a staging path before writing.

SECTORS = {
    "Financial Services": {
        "names": ["Asterion Payments", "Bruma Credit", "Cedra Ledger", "Dorado Capital", "Echelon Remit", "Faro Wealth", "Granito Risk", "Horizonte Leasing", "Ibis Treasury", "Juniper Exchange"],
        "subsectors": ["Digital payments", "SME lending", "Banking software", "Specialty finance", "Cross-border remittances", "Wealth technology", "Insurance analytics", "Equipment leasing", "Treasury management", "Market infrastructure"],
        "products": ["merchant acquiring and embedded payment rails", "data-driven working-capital loans", "cloud-native core banking software", "asset-backed specialty credit", "cross-border settlement services", "digital portfolio and advisory tools", "risk-pricing software for insurers", "equipment and vehicle financing", "cash-management and liquidity software", "electronic trading and post-trade infrastructure"],
        "customers": ["mid-market merchants", "small and medium-sized enterprises", "regional financial institutions", "specialty distributors", "migrant workers and payment providers", "wealth managers and affluent households", "property and casualty insurers", "industrial and logistics operators", "corporate treasury teams", "brokers, exchanges and asset managers"],
        "models": ["Transaction fees", "Net interest income", "Subscription", "Interest spread", "Transaction fees", "Assets under management fees", "Subscription", "Interest spread", "Subscription", "Transaction fees"],
        "theme": "Financial Inclusion",
    },
    "Technology": {
        "names": ["KairoStack", "LuminaGrid", "MeridianAI", "NimbusForge", "OrigoCloud", "PrismaLogic", "QuantaWorks", "RivetSoft", "SignalArc", "TensorPeak"],
        "subsectors": ["Developer tools", "Data infrastructure", "Enterprise AI", "Cybersecurity", "Cloud operations", "Workflow automation", "Industrial software", "Vertical SaaS", "Communications software", "AI infrastructure"],
        "products": ["application development and observability tools", "distributed data orchestration software", "enterprise decision-intelligence models", "identity and threat-detection software", "multi-cloud cost and reliability management", "document and process automation", "factory planning and digital-twin software", "specialized software for professional services", "secure communications and contact-center tools", "model training and inference infrastructure"],
        "customers": ["software engineering teams", "data-intensive enterprises", "large corporate functions", "regulated organizations", "cloud-native companies", "operations and finance teams", "manufacturers", "professional-services firms", "customer-service organizations", "AI developers and research laboratories"],
        "models": ["Subscription"] * 10,
        "theme": "Digital Infrastructure",
    },
    "Healthcare": {
        "names": ["Alba Diagnostics", "Biomea Care", "CuraNova", "DeltaMed Systems", "Eirene Clinics", "FloraGen", "Helix Harbor", "Ionis Devices", "Jade Therapeutics", "Kinetica Health"],
        "subsectors": ["Diagnostics", "Virtual care", "Specialty clinics", "Hospital software", "Outpatient care", "Genomics", "Clinical research", "Medical devices", "Biotechnology", "Rehabilitation"],
        "products": ["rapid diagnostic platforms", "virtual chronic-care programs", "specialized oncology clinics", "hospital administration and clinical software", "outpatient surgical centers", "genomic testing and decision support", "clinical-trial operations services", "minimally invasive medical devices", "targeted therapies for rare diseases", "technology-enabled physical rehabilitation"],
        "customers": ["laboratories and hospitals", "employers and health plans", "patients and insurers", "hospitals and health systems", "patients and referring physicians", "clinicians and pharmaceutical companies", "biopharma sponsors", "surgeons and hospitals", "specialist clinicians and patients", "employers, insurers and patients"],
        "models": ["Per-test revenue", "Subscription", "Fee for service", "Subscription", "Fee for service", "Per-test revenue", "Contract services", "Product sales", "Licensing and milestones", "Fee for service"],
        "theme": "Aging Population",
    },
    "Industrials": {
        "names": ["ArcForge Industries", "Boreal Automation", "Cinder Components", "Dynamo Controls", "Estela Robotics", "FerroMotion", "Galen Materials", "HelioMach", "Iberia Systems", "Juno Precision"],
        "subsectors": ["Engineered products", "Factory automation", "Electronic components", "Industrial controls", "Robotics", "Motion systems", "Advanced materials", "Capital equipment", "Industrial systems", "Precision manufacturing"],
        "products": ["engineered products for harsh environments", "factory automation platforms", "high-reliability electronic components", "process-control and sensing systems", "collaborative industrial robots", "motors, drives and motion components", "specialty materials and coatings", "high-efficiency production equipment", "integrated systems for infrastructure operators", "precision parts for aerospace and medical applications"],
        "customers": ["energy and infrastructure operators", "manufacturers", "electronics and mobility manufacturers", "process industries", "mid-sized manufacturers", "equipment manufacturers", "industrial producers", "manufacturing plants", "utilities and transportation authorities", "aerospace and medical-device companies"],
        "models": ["Product sales", "Product and service", "Product sales", "Product and service", "Product sales", "Product sales", "Product sales", "Product and service", "Project revenue", "Product sales"],
        "theme": "Supply Chain Resilience",
    },
    "Consumer": {
        "names": ["Amapola Living", "Basil & Stone", "Casa Nativa", "Dulce Norte", "Everfield Apparel", "Fable House", "Gala Beauty", "Hearth & Pine", "Isla Active", "Jardín Market"],
        "subsectors": ["Home products", "Premium food", "Lifestyle retail", "Confectionery", "Apparel", "Home furnishings", "Beauty", "Consumer durables", "Fitness apparel", "Specialty grocery"],
        "products": ["sustainable home and lifestyle products", "premium packaged foods", "omnichannel lifestyle retail", "branded confectionery and snacks", "everyday apparel", "design-led furniture and accessories", "skin-care and personal-care products", "small home appliances", "technical fitness apparel", "fresh and specialty grocery retail"],
        "customers": ["urban households", "premium grocery consumers", "middle-income households", "mass-market consumers", "value-conscious consumers", "homeowners and renters", "beauty consumers", "households", "fitness-oriented consumers", "urban households"],
        "models": ["Omnichannel retail", "Wholesale and DTC", "Retail", "Wholesale", "Wholesale and DTC", "Retail", "DTC and wholesale", "Product sales", "DTC and wholesale", "Retail"],
        "theme": "Consumer Premiumization",
    },
    "Energy and Utilities": {
        "names": ["Aureus Power", "BlueCurrent", "CarbonVale", "DawnGrid", "Ember Storage", "Flux Renewables", "Gaia Utilities", "Hydra Networks", "Ionic Energy", "Keystone Fuels"],
        "subsectors": ["Renewable generation", "Water infrastructure", "Carbon management", "Grid technology", "Energy storage", "Distributed energy", "Regulated utilities", "Transmission", "Battery systems", "Conventional fuels"],
        "products": ["utility-scale solar and wind generation", "water treatment and distribution systems", "industrial emissions measurement and abatement", "grid monitoring and demand-management software", "battery storage projects", "distributed solar and energy services", "regulated electricity and water service", "high-voltage transmission infrastructure", "battery modules and energy-management systems", "refined fuel distribution and logistics"],
        "customers": ["utilities and corporate buyers", "municipalities and industrial users", "heavy industry", "utilities and grid operators", "utilities and renewable developers", "commercial and industrial facilities", "residential and business customers", "utilities and system operators", "mobility and industrial customers", "commercial fleets and distributors"],
        "models": ["Long-term contracts", "Project and service revenue", "Subscription and project", "Subscription", "Long-term contracts", "Energy-as-a-service", "Regulated tariffs", "Regulated and contracted", "Product sales", "Distribution margin"],
        "theme": "Decarbonization",
    },
    "Real Estate": {
        "names": ["Altura Logistics RE", "Bosque Living", "Cívica Offices", "Distrito Data Centers", "Estación Retail", "Forma Industrial", "Gran Via Hospitality", "Hábitat Senior", "Índigo Storage", "Ladera Communities"],
        "subsectors": ["Logistics real estate", "Multifamily", "Offices", "Data centers", "Retail centers", "Industrial parks", "Hospitality", "Senior housing", "Self-storage", "Residential development"],
        "products": ["modern logistics facilities", "professionally managed rental housing", "premium office portfolios", "carrier-neutral data centers", "grocery-anchored retail centers", "industrial and light-manufacturing parks", "urban and resort hotels", "senior-living communities", "self-storage facilities", "master-planned residential communities"],
        "customers": ["e-commerce and logistics tenants", "urban renters", "corporate tenants", "cloud and enterprise customers", "retailers and local consumers", "manufacturers and distributors", "business and leisure travelers", "older adults and families", "households and small businesses", "homebuyers"],
        "models": ["Rental income", "Rental income", "Rental income", "Colocation contracts", "Rental income", "Rental income", "Room and service revenue", "Rental and service income", "Rental income", "Development sales"],
        "theme": "Digital Infrastructure",
    },
    "Telecom and Media": {
        "names": ["Auralink Networks", "Beacon Fiber", "Croma Studios", "Dialtone Mobile", "EchoWave Media", "Futura Towers", "Glyph Content", "Halo Broadband", "Ícaro Streaming", "Junction Ads"],
        "subsectors": ["Enterprise connectivity", "Fiber networks", "Content production", "Mobile services", "Digital media", "Telecom towers", "Publishing technology", "Broadband", "Streaming", "Advertising technology"],
        "products": ["managed connectivity and communications", "metropolitan and regional fiber networks", "scripted and unscripted audiovisual content", "mobile connectivity for value segments", "digital news and specialist media", "wireless infrastructure and tower leasing", "digital publishing and rights-management tools", "fixed wireless and fiber broadband", "regional entertainment streaming", "programmatic advertising infrastructure"],
        "customers": ["enterprises", "carriers and enterprises", "broadcasters and streaming platforms", "consumers and small businesses", "professionals and advertisers", "mobile-network operators", "publishers and creators", "households and small businesses", "households", "brands, agencies and publishers"],
        "models": ["Subscription", "Long-term contracts", "Production fees and rights", "Subscription", "Subscription and advertising", "Long-term leases", "Subscription", "Subscription", "Subscription", "Transaction fees"],
        "theme": "Digital Infrastructure",
    },
    "Mobility and Logistics": {
        "names": ["Atlas Freight", "Borealis Mobility", "CargoLynx", "Drift Transit", "Estafeta Urbana", "FleetNova", "Glide Aviation", "HarborChain", "Intermodal One", "Jetstream Services"],
        "subsectors": ["Freight brokerage", "Electric mobility", "Logistics software", "Urban transit", "Last-mile delivery", "Fleet management", "Aviation services", "Port logistics", "Intermodal transport", "Aircraft services"],
        "products": ["technology-enabled freight brokerage", "electric commercial vehicles", "transport management and visibility software", "contracted urban transportation", "same-day and next-day delivery", "fleet telematics and maintenance services", "ground handling and aviation support", "port and maritime logistics", "rail and truck intermodal services", "aircraft maintenance and component services"],
        "customers": ["manufacturers and retailers", "commercial fleets", "shippers and logistics providers", "cities and large employers", "retailers and e-commerce platforms", "commercial fleets", "airlines and airports", "importers, exporters and carriers", "industrial shippers", "airlines and aircraft owners"],
        "models": ["Transaction margin", "Product and service", "Subscription", "Long-term contracts", "Per-delivery fees", "Subscription and service", "Service revenue", "Service revenue", "Contract services", "Service revenue"],
        "theme": "Supply Chain Resilience",
    },
    "Agriculture and Food": {
        "names": ["Agave Fields", "BioHarvest", "Campo Claro", "Delta Protein", "EcoRiego", "FincaNova", "GrainBridge", "Huerto Labs", "Ibis Foods", "Koru Ingredients"],
        "subsectors": ["Specialty crops", "Agricultural inputs", "Farm management", "Alternative protein", "Irrigation technology", "Controlled-environment agriculture", "Grain trading", "Crop science", "Packaged foods", "Food ingredients"],
        "products": ["premium and traceable specialty crops", "biological crop-protection products", "farm-management and procurement services", "plant-based protein products", "precision irrigation systems", "greenhouse-grown produce", "grain origination and distribution", "seed analytics and crop-science services", "branded convenient foods", "specialty ingredients for food manufacturers"],
        "customers": ["food processors and exporters", "farmers and distributors", "small and medium-sized farms", "retailers and food-service companies", "farmers", "retailers and food-service companies", "food processors and industrial buyers", "seed companies and farmers", "mass-market consumers", "food and beverage manufacturers"],
        "models": ["Product sales", "Product sales", "Service and transaction", "Wholesale and DTC", "Product and service", "Contract sales", "Trading margin", "Licensing and service", "Wholesale", "Product sales"],
        "theme": "Supply Chain Resilience",
    },
}

STYLES = (["Hypergrowth"] * 15 + ["Growth"] * 20 + ["Quality Compounder"] * 20 +
          ["Value"] * 15 + ["Turnaround"] * 15 + ["No Growth"] * 15)
random.shuffle(STYLES)

REGIONS = ["North America", "Latin America", "Europe", "Asia-Pacific", "Middle East and Africa"]
OWNERSHIP = {
    "Startup": ["Founder-led", "VC-backed"],
    "Scale-up": ["VC-backed", "Growth-equity backed", "Founder-led"],
    "Established": ["Public", "Private equity-backed", "Family-owned", "Founder-led"],
    "Mature": ["Public", "Family-owned", "Private equity-backed"],
}


def clean_file_name(value: str) -> str:
    return re.sub(r'[\\/:*?"<>|]', "-", value)


def yaml_quote(value: str) -> str:
    return json.dumps(value, ensure_ascii=False)


def lifecycle(style: str, idx: int) -> str:
    if style == "Hypergrowth":
        return "Startup" if idx % 2 == 0 else "Scale-up"
    if style == "Growth":
        return "Scale-up" if idx % 3 else "Established"
    if style == "Quality Compounder":
        return "Established" if idx % 3 else "Mature"
    if style == "Value":
        return "Mature" if idx % 2 else "Established"
    if style == "Turnaround":
        return "Established" if idx % 2 else "Mature"
    return "Mature"


def revenue_range(stage: str) -> tuple[float, float]:
    return {
        "Startup": (8, 70),
        "Scale-up": (60, 650),
        "Established": (450, 4200),
        "Mature": (1800, 16000),
    }[stage]


def growth_for(style: str) -> float:
    ranges = {
        "Hypergrowth": (38, 82), "Growth": (16, 37), "Quality Compounder": (8, 18),
        "Value": (-2, 9), "Turnaround": (-12, 8), "No Growth": (-5, 3),
    }
    return round(random.uniform(*ranges[style]), 1)


def margin_for(style: str, stage: str, sector: str) -> float:
    ranges = {
        "Hypergrowth": (-34, 8), "Growth": (-5, 18), "Quality Compounder": (16, 34),
        "Value": (10, 25), "Turnaround": (-10, 12), "No Growth": (7, 22),
    }
    margin = random.uniform(*ranges[style])
    if sector == "Real Estate":
        margin += 22
    if sector in {"Consumer", "Mobility and Logistics", "Agriculture and Food"}:
        margin -= 3
    if stage == "Startup":
        margin -= 4
    return round(max(-42, min(62, margin)), 1)


def transaction_angle(stage: str, style: str, leverage: float) -> tuple[str, str]:
    if stage == "Startup":
        return "Growth capital", "Minority capital raise to fund product expansion and commercial scale-up."
    if stage == "Scale-up":
        return "Pre-IPO or growth equity", "Institutional growth financing with governance enhancement and IPO-readiness work."
    if style == "Turnaround":
        return "Restructuring or carve-out", "Operational restructuring, liability management and possible sale of non-core assets."
    if style == "Value":
        return "Buyout or strategic review", "Potential sponsor-led buyout, take-private or strategic alternatives process."
    if leverage > 3.5:
        return "Debt refinancing", "Refinancing and maturity extension supported by a deleveraging plan."
    if stage == "Mature":
        return "Strategic M&A", "Platform acquisition, divestiture or consolidation opportunity within the sector."
    return "Sell-side or add-on M&A", "Potential strategic sale or acquisition program to accelerate sector consolidation."


def make_company_records() -> list[dict]:
    records = []
    global_index = 0
    for sector_index, (sector, cfg) in enumerate(SECTORS.items()):
        for local_index, name in enumerate(cfg["names"]):
            style = STYLES[global_index]
            stage = lifecycle(style, global_index)
            region = REGIONS[(global_index + sector_index) % len(REGIONS)]
            lo, hi = revenue_range(stage)
            revenue = round(random.uniform(lo, hi), 1)
            growth = growth_for(style)
            prev_revenue = round(revenue / (1 + growth / 100), 1)
            margin = margin_for(style, stage, sector)
            ebitda = round(revenue * margin / 100, 1)
            cash_pct = random.uniform(0.04, 0.30) if stage in {"Startup", "Scale-up"} else random.uniform(0.02, 0.12)
            cash = round(revenue * cash_pct, 1)
            debt_factor = random.uniform(0.0, 0.20) if stage == "Startup" else random.uniform(0.05, 0.75)
            if style in {"Value", "Turnaround", "No Growth"}:
                debt_factor += random.uniform(0.15, 0.65)
            debt = round(revenue * debt_factor, 1)
            net_debt = round(debt - cash, 1)
            capex_pct = random.uniform(0.03, 0.09)
            if sector in {"Energy and Utilities", "Real Estate", "Telecom and Media", "Industrials"}:
                capex_pct += random.uniform(0.04, 0.12)
            capex = round(revenue * capex_pct, 1)
            fcf = round(ebitda - capex - max(0, ebitda) * random.uniform(0.10, 0.24) - revenue * random.uniform(-0.01, 0.035), 1)
            net_income = round(ebitda - max(0, debt * random.uniform(0.035, 0.09)) - capex * random.uniform(0.25, 0.55) - max(0, ebitda) * random.uniform(0.08, 0.20), 1)
            if ebitda > 0:
                ev_ebitda = {
                    "Hypergrowth": random.uniform(18, 32), "Growth": random.uniform(13, 24),
                    "Quality Compounder": random.uniform(12, 22), "Value": random.uniform(5, 10),
                    "Turnaround": random.uniform(4, 9), "No Growth": random.uniform(5, 9),
                }[style]
                ev = round(ebitda * ev_ebitda, 1)
            else:
                ev_revenue = random.uniform(4.5, 11) if style == "Hypergrowth" else random.uniform(1.5, 5)
                ev = round(revenue * ev_revenue, 1)
            ev = round(max(ev, revenue * 0.45, debt - cash + 5), 1)
            equity_value = round(ev - debt + cash, 1)
            ev_revenue = round(ev / revenue, 2)
            ev_ebitda = round(ev / ebitda, 2) if ebitda > 0 else None
            pe = round(equity_value / net_income, 2) if net_income > 0 else None
            leverage = round(net_debt / ebitda, 2) if ebitda > 0 else None
            employees = int(max(25, revenue * random.uniform(1.3, 5.8)))
            founded = random.randint(2015, 2023) if stage == "Startup" else random.randint(2005, 2017) if stage == "Scale-up" else random.randint(1970, 2012)
            recurring_base = 82 if "Subscription" in cfg["models"][local_index] or "Rental" in cfg["models"][local_index] or "Long-term" in cfg["models"][local_index] else 38
            recurring = int(max(5, min(98, random.gauss(recurring_base, 13))))
            concentration = int(max(5, min(58, random.gauss(24 if stage in {"Startup", "Scale-up"} else 16, 9))))
            moat = int(max(1, min(10, round(random.gauss(7 if style in {"Hypergrowth", "Quality Compounder"} else 5.5, 1.4)))))
            management = int(max(1, min(10, round(random.gauss(7, 1.3)))))
            risk = int(max(1, min(10, round(random.gauss(7.5 if style in {"Hypergrowth", "Turnaround"} else 5, 1.4)))))
            esg = int(max(20, min(95, round(random.gauss(66, 13)))))
            roic = round(random.uniform(-25, 4), 1) if ebitda <= 0 else round(random.uniform(4, 26), 1)
            tx_type, tx_rationale = transaction_angle(stage, style, leverage if leverage is not None else 0)
            ownership = random.choice(OWNERSHIP[stage])
            status = "Profitable" if net_income > 0 else "Loss-making"
            market_position = random.choice(["Niche leader", "Top-three regional player", "Emerging challenger", "Scaled incumbent", "Specialist operator"])
            description = (
                f"{name} is a fictional {cfg['subsectors'][local_index].lower()} company serving {cfg['customers'][local_index]}. "
                f"It provides {cfg['products'][local_index]} through a {cfg['models'][local_index].lower()} model. "
                f"The company operates primarily in {region} and is positioned as a {market_position.lower()}."
            )
            records.append({
                "id": f"SYN-{global_index + 1:03d}", "name": name, "sector": sector,
                "subsector": cfg["subsectors"][local_index], "theme": cfg["theme"], "region": region,
                "stage": stage, "investment_style": style, "ownership": ownership,
                "founded": founded, "employees": employees, "business_model": cfg["models"][local_index],
                "market_position": market_position, "description": description,
                "revenue_prev_usd_m": prev_revenue, "revenue_usd_m": revenue, "revenue_growth_pct": growth,
                "ebitda_usd_m": ebitda, "ebitda_margin_pct": margin, "net_income_usd_m": net_income,
                "free_cash_flow_usd_m": fcf, "cash_usd_m": cash, "debt_usd_m": debt,
                "net_debt_usd_m": net_debt, "enterprise_value_usd_m": ev,
                "equity_value_usd_m": equity_value, "ev_revenue": ev_revenue,
                "ev_ebitda": ev_ebitda, "pe_ratio": pe, "net_leverage": leverage,
                "roic_pct": roic, "recurring_revenue_pct": recurring,
                "top_customer_concentration_pct": concentration, "moat_score": moat,
                "management_score": management, "execution_risk_score": risk, "esg_score": esg,
                "profitability": status, "transaction_type": tx_type, "transaction_rationale": tx_rationale,
            })
            global_index += 1
    return records


def frontmatter(record: dict, comparables: list[str]) -> str:
    fields = [
        ("type", "company"), ("synthetic", True), ("company_id", record["id"]),
        ("company", record["name"]), ("sector", record["sector"]), ("subsector", record["subsector"]),
        ("theme", record["theme"]), ("region", record["region"]), ("stage", record["stage"]),
        ("investment_style", record["investment_style"]), ("ownership", record["ownership"]),
        ("founded", record["founded"]), ("employees", record["employees"]),
        ("business_model", record["business_model"]), ("market_position", record["market_position"]),
        ("revenue_prev_usd_m", record["revenue_prev_usd_m"]), ("revenue_usd_m", record["revenue_usd_m"]),
        ("revenue_growth_pct", record["revenue_growth_pct"]), ("ebitda_usd_m", record["ebitda_usd_m"]),
        ("ebitda_margin_pct", record["ebitda_margin_pct"]), ("net_income_usd_m", record["net_income_usd_m"]),
        ("free_cash_flow_usd_m", record["free_cash_flow_usd_m"]), ("cash_usd_m", record["cash_usd_m"]),
        ("debt_usd_m", record["debt_usd_m"]), ("net_debt_usd_m", record["net_debt_usd_m"]),
        ("enterprise_value_usd_m", record["enterprise_value_usd_m"]), ("equity_value_usd_m", record["equity_value_usd_m"]),
        ("ev_revenue", record["ev_revenue"]), ("ev_ebitda", record["ev_ebitda"]),
        ("pe_ratio", record["pe_ratio"]), ("net_leverage", record["net_leverage"]),
        ("roic_pct", record["roic_pct"]), ("recurring_revenue_pct", record["recurring_revenue_pct"]),
        ("top_customer_concentration_pct", record["top_customer_concentration_pct"]),
        ("moat_score", record["moat_score"]), ("management_score", record["management_score"]),
        ("execution_risk_score", record["execution_risk_score"]), ("esg_score", record["esg_score"]),
        ("profitability", record["profitability"]), ("transaction_type", record["transaction_type"]),
        ("comparables", comparables), ("tags", ["company", "synthetic-data", clean_file_name(record["sector"]).lower().replace(" ", "-")]),
    ]
    lines = ["---"]
    for key, value in fields:
        if value is None:
            lines.append(f"{key}: null")
        elif isinstance(value, bool):
            lines.append(f"{key}: {'true' if value else 'false'}")
        elif isinstance(value, (int, float)):
            lines.append(f"{key}: {value}")
        elif isinstance(value, list):
            lines.append(f"{key}: [{', '.join(yaml_quote(str(x)) for x in value)}]")
        else:
            lines.append(f"{key}: {yaml_quote(str(value))}")
    lines.append("---")
    return "\n".join(lines)


def fmt(value, suffix="") -> str:
    if value is None:
        return "n.m."
    return f"{value:,.2f}{suffix}" if isinstance(value, float) else f"{value}{suffix}"


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.rstrip() + "\n", encoding="utf-8")


def create_vault(records: list[dict]) -> None:
    if ROOT.exists():
        shutil.rmtree(ROOT)
    for folder in [
        "Companies", "Sectors", "Investment Profiles", "Themes", "Transactions", "Maps",
        "Dashboards", "Data", "Templates", "wiki", ".obsidian", "Environment", "Scenarios",
        "Relationships", "Opportunities", "Decisions", "Daily Briefs", "Audit", "Notebooks",
        "Sources", "Claims", "Evidence",
    ]:
        (ROOT / folder).mkdir(parents=True, exist_ok=True)

    by_sector = defaultdict(list)
    by_style = defaultdict(list)
    by_theme = defaultdict(list)
    for record in records:
        by_sector[record["sector"]].append(record)
        by_style[record["investment_style"]].append(record)
        by_theme[record["theme"]].append(record)

    for sector, companies in by_sector.items():
        for i, record in enumerate(companies):
            peers = [companies[(i + offset) % len(companies)]["name"] for offset in (1, 2, 3)]
            peer_links = [f"[[Companies/{clean_file_name(p)}|{p}]]" for p in peers]
            fm = frontmatter(record, peers)
            body = f"""{fm}

# {record['name']}

> [!warning] Synthetic company
> This company and every figure in this note are fictional and intended solely for investment-banking training, analytics and workflow design.

## Investment snapshot

| Dimension | Assessment |
|---|---|
| Sector | [[Sectors/{clean_file_name(record['sector'])}|{record['sector']}]] · {record['subsector']} |
| Profile | [[Investment Profiles/{clean_file_name(record['investment_style'])}|{record['investment_style']}]] · {record['stage']} |
| Strategic theme | [[Themes/{clean_file_name(record['theme'])}|{record['theme']}]] |
| Geography | {record['region']} |
| Ownership | {record['ownership']} |
| Position | {record['market_position']} |
| Transaction angle | **{record['transaction_type']}** |

## Business description

{record['description']}

### Revenue model

- **Model:** {record['business_model']}
- **Recurring revenue:** {record['recurring_revenue_pct']}%
- **Top-customer concentration:** {record['top_customer_concentration_pct']}%
- **Employees:** {record['employees']:,}
- **Founded:** {record['founded']}

## Selected financials

All monetary figures are synthetic USD millions.

| Metric | Prior year | Current year |
|---|---:|---:|
| Revenue | {record['revenue_prev_usd_m']:,.1f} | {record['revenue_usd_m']:,.1f} |
| Revenue growth | — | {record['revenue_growth_pct']:,.1f}% |
| EBITDA | — | {record['ebitda_usd_m']:,.1f} |
| EBITDA margin | — | {record['ebitda_margin_pct']:,.1f}% |
| Net income | — | {record['net_income_usd_m']:,.1f} |
| Free cash flow | — | {record['free_cash_flow_usd_m']:,.1f} |
| Cash | — | {record['cash_usd_m']:,.1f} |
| Debt | — | {record['debt_usd_m']:,.1f} |
| Net debt | — | {record['net_debt_usd_m']:,.1f} |

## Valuation

| Metric | Value |
|---|---:|
| Enterprise value | ${record['enterprise_value_usd_m']:,.1f}m |
| Equity value | ${record['equity_value_usd_m']:,.1f}m |
| EV / Revenue | {fmt(record['ev_revenue'], 'x')} |
| EV / EBITDA | {fmt(record['ev_ebitda'], 'x')} |
| P / E | {fmt(record['pe_ratio'], 'x')} |
| Net leverage | {fmt(record['net_leverage'], 'x')} |
| ROIC | {record['roic_pct']:,.1f}% |

## Qualitative scorecard

| Factor | Score |
|---|---:|
| Competitive moat | {record['moat_score']}/10 |
| Management quality | {record['management_score']}/10 |
| Execution risk | {record['execution_risk_score']}/10 |
| ESG score | {record['esg_score']}/100 |

## Transaction thesis

**Recommended lens:** {record['transaction_type']}.

{record['transaction_rationale']}

### Preliminary diligence questions

1. How defensible are growth, margins and customer-retention assumptions?
2. Which operational or regulatory factors could alter normalized EBITDA?
3. What is the sustainable capital structure under downside conditions?
4. Which strategic or financial buyers could realize identifiable synergies?
5. What information would be required before moving from screening to valuation?

## Comparable companies

""" + "\n".join(f"- {link}" for link in peer_links) + f"""

## Related maps

- [[Maps/Market Landscape|Market Landscape]]
- [[Transactions/Transaction Opportunities|Transaction Opportunities]]
- [[Dashboards/Investment Banking Dashboard|Investment Banking Dashboard]]
"""
            write_text(ROOT / "Companies" / f"{clean_file_name(record['name'])}.md", body)

    for sector, companies in by_sector.items():
        rows = []
        for r in sorted(companies, key=lambda x: x["enterprise_value_usd_m"], reverse=True):
            rows.append(f"| [[Companies/{clean_file_name(r['name'])}|{r['name']}]] | {r['subsector']} | {r['stage']} | {r['investment_style']} | {r['revenue_growth_pct']:.1f}% | {r['ebitda_margin_pct']:.1f}% | {r['enterprise_value_usd_m']:,.1f} | {fmt(r['ev_ebitda'], 'x')} | {r['transaction_type']} |")
        content = f"""---
type: sector
sector: {yaml_quote(sector)}
company_count: {len(companies)}
tags: [sector, synthetic-data]
---

# {sector}

## Sector screen

| Company | Subsector | Stage | Profile | Growth | EBITDA margin | EV $m | EV/EBITDA | Transaction angle |
|---|---|---|---|---:|---:|---:|---:|---|
{chr(10).join(rows)}

## Strategic theme

Primary connection: [[Themes/{clean_file_name(SECTORS[sector]['theme'])}|{SECTORS[sector]['theme']}]].

## Sector map

Open [[Maps/{clean_file_name(sector)} Map|{sector} Map]].
"""
        write_text(ROOT / "Sectors" / f"{clean_file_name(sector)}.md", content)

    for style, companies in by_style.items():
        rows = [f"- [[Companies/{clean_file_name(r['name'])}|{r['name']}]] — {r['sector']}; growth {r['revenue_growth_pct']:.1f}%; EBITDA margin {r['ebitda_margin_pct']:.1f}%" for r in sorted(companies, key=lambda x: x["revenue_growth_pct"], reverse=True)]
        content = f"""---
type: investment-profile
profile: {yaml_quote(style)}
company_count: {len(companies)}
tags: [investment-profile, synthetic-data]
---

# {style}

This profile groups fictional companies by their dominant investment characteristics. It is a screening category, not an investment recommendation.

## Companies

{chr(10).join(rows)}
"""
        write_text(ROOT / "Investment Profiles" / f"{clean_file_name(style)}.md", content)

    for theme, companies in by_theme.items():
        rows = [f"- [[Companies/{clean_file_name(r['name'])}|{r['name']}]] — [[Sectors/{clean_file_name(r['sector'])}|{r['sector']}]]" for r in companies]
        content = f"""---
type: theme
theme: {yaml_quote(theme)}
company_count: {len(companies)}
tags: [strategic-theme, synthetic-data]
---

# {theme}

## Connected companies

{chr(10).join(rows)}

## Banking questions

- Where is capital formation most likely?
- Which companies are natural consolidators or targets?
- Which valuation metrics best capture economics across sectors?
- What regulatory, technological or capital-cycle risks could disrupt the thesis?
"""
        write_text(ROOT / "Themes" / f"{clean_file_name(theme)}.md", content)

    opportunities = defaultdict(list)
    for r in records:
        opportunities[r["transaction_type"]].append(r)
    sections = []
    for tx_type, companies in sorted(opportunities.items()):
        lines = [f"- [[Companies/{clean_file_name(r['name'])}|{r['name']}]] — {r['sector']}; EV ${r['enterprise_value_usd_m']:,.1f}m; {r['investment_style']}" for r in sorted(companies, key=lambda x: x["enterprise_value_usd_m"], reverse=True)]
        sections.append(f"## {tx_type}\n\n" + "\n".join(lines))
    write_text(ROOT / "Transactions" / "Transaction Opportunities.md", """---
type: transaction-pipeline
synthetic: true
tags: [transactions, investment-banking, synthetic-data]
---

# Transaction Opportunities

> [!warning] Training pipeline
> Every company and opportunity is fictional. This is a banking-workflow prototype, not a recommendation or live mandate.

""" + "\n\n".join(sections))

    # CSV master data
    csv_fields = list(records[0].keys())
    with (ROOT / "Data" / "company_master.csv").open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=csv_fields)
        writer.writeheader()
        writer.writerows(records)

    # Obsidian Bases configuration
    write_text(ROOT / "Dashboards" / "Company Screener.base", """filters:
  and:
    - file.hasTag("company")
views:
  - type: table
    name: All companies
    order:
      - file.name
      - sector
      - stage
      - investment_style
      - revenue_usd_m
      - revenue_growth_pct
      - ebitda_margin_pct
      - enterprise_value_usd_m
      - ev_revenue
      - ev_ebitda
      - transaction_type
  - type: table
    name: Growth screen
    filters:
      and:
        - revenue_growth_pct >= 15
    order:
      - file.name
      - sector
      - revenue_growth_pct
      - ebitda_margin_pct
      - enterprise_value_usd_m
  - type: table
    name: Value and turnaround
    filters:
      or:
        - investment_style == "Value"
        - investment_style == "Turnaround"
    order:
      - file.name
      - investment_style
      - ev_ebitda
      - net_leverage
      - free_cash_flow_usd_m
""")

    dashboard = """---
type: dashboard
tags: [dashboard, investment-banking, synthetic-data]
---

# Investment Banking Dashboard

> [!important] Start here
> This vault contains 100 fictional companies designed for investment-banking research, valuation, relationship mapping and transaction-screening exercises.

## Core views

- [[Dashboards/Company Screener.base|Interactive Company Screener]]
- [[Maps/Market Landscape|Market Landscape Canvas]]
- [[Transactions/Transaction Opportunities|Transaction Opportunities]]
- [[Data/Data Dictionary|Data Dictionary]]
- [[wiki/hot|Hot Context Cache]]

## Sector maps

""" + "\n".join(f"- [[Sectors/{clean_file_name(s)}|{s}]] · [[Maps/{clean_file_name(s)} Map|Canvas]]" for s in SECTORS) + """

## Investment profiles

""" + "\n".join(f"- [[Investment Profiles/{clean_file_name(s)}|{s}]]" for s in ["Hypergrowth", "Growth", "Quality Compounder", "Value", "Turnaround", "No Growth"]) + """

## Optional Dataview screen

```dataview
TABLE sector, stage, investment_style, revenue_usd_m, revenue_growth_pct,
ebitda_margin_pct, enterprise_value_usd_m, ev_ebitda, transaction_type
FROM #company
SORT enterprise_value_usd_m DESC
```
"""
    write_text(ROOT / "Dashboards" / "Investment Banking Dashboard.md", dashboard)
    write_text(ROOT / "00 Home.md", """---
type: home
vault: Alejandro-Reynoso-Investment-Banking-Vault
synthetic: true
tags: [home, investment-banking, synthetic-data]
---

# Alejandro Reynoso — Investment Banking Vault

Open the [[Dashboards/Investment Banking Dashboard|Investment Banking Dashboard]].

This is a standalone Obsidian vault containing 100 fictional companies across ten sectors. It supports company screening, fundamental analysis, valuation comparisons, sector mapping, transaction origination and investment-profile classification.

> [!warning] Synthetic dataset
> No company, security, valuation or transaction in this vault represents real-world information or investment advice.
""")

    dictionary_rows = [
        ("revenue_usd_m", "Current-year revenue", "USD millions"),
        ("revenue_growth_pct", "Year-over-year revenue growth", "%"),
        ("ebitda_usd_m", "Earnings before interest, tax, depreciation and amortization", "USD millions"),
        ("ebitda_margin_pct", "EBITDA divided by revenue", "%"),
        ("free_cash_flow_usd_m", "Synthetic post-capex free cash flow", "USD millions"),
        ("enterprise_value_usd_m", "Equity value plus debt less cash", "USD millions"),
        ("ev_revenue", "Enterprise value divided by revenue", "x"),
        ("ev_ebitda", "Enterprise value divided by positive EBITDA", "x or null"),
        ("pe_ratio", "Equity value divided by positive net income", "x or null"),
        ("net_leverage", "Net debt divided by positive EBITDA", "x or null"),
        ("moat_score", "Illustrative competitive-position score", "1–10"),
        ("execution_risk_score", "Illustrative operating and execution risk", "1–10; higher is riskier"),
        ("esg_score", "Illustrative ESG score", "0–100"),
    ]
    dd = "\n".join(f"| `{a}` | {b} | {c} |" for a, b, c in dictionary_rows)
    write_text(ROOT / "Data" / "Data Dictionary.md", f"""# Data Dictionary

All values are synthetic and denominated in USD unless otherwise stated.

| Field | Meaning | Unit |
|---|---|---|
{dd}

Null valuation multiples mean that the relevant denominator is zero or negative and the multiple is not meaningful.
""")

    write_text(ROOT / "Templates" / "Company Template.md", """---
type: company
synthetic: false
company_id:
company:
sector:
subsector:
theme:
region:
stage:
investment_style:
ownership:
revenue_usd_m:
revenue_growth_pct:
ebitda_usd_m:
ebitda_margin_pct:
enterprise_value_usd_m:
transaction_type:
tags: [company]
---

# {{title}}

## Business description

## Financials

## Valuation

## Transaction thesis

## Risks and diligence questions

## Comparable companies
""")

    write_text(ROOT / "wiki" / "hot.md", """# Hot Context Cache

## Current state

- Standalone synthetic investment-banking vault created with 100 fictional companies.
- Ten sectors, six investment profiles and six strategic themes are populated.
- Company notes contain operating fundamentals, valuation metrics, qualitative scores, transaction angles and comparable-company links.

## Recommended next actions

1. Open [[00 Home]].
2. Explore [[Dashboards/Investment Banking Dashboard]].
3. Filter the [[Dashboards/Company Screener.base|Company Screener]].
4. Select one sector and develop a mock pitch, valuation or transaction mandate.

## Guardrail

All information is synthetic and must not be represented as market data or investment advice.
""")

    write_text(ROOT / "README.md", """# Alejandro-Reynoso-Investment-Banking-Vault

Standalone Obsidian vault containing 100 fictional companies for investment-banking workflow design.

Start with `00 Home.md`. The vault works with core Obsidian. The `.base` screener uses Obsidian Bases; the optional Dataview block requires the Dataview community plugin.

All companies and figures are synthetic.
""")

    write_text(ROOT / ".obsidian" / "app.json", json.dumps({"showUnsupportedFiles": True, "alwaysUpdateLinks": True, "newFileLocation": "folder", "newFileFolderPath": "Companies"}, indent=2))
    write_text(ROOT / ".obsidian" / "graph.json", json.dumps({"collapse-filter": False, "search": "-path:Templates", "showTags": True, "showAttachmentsOnly": False, "hideUnresolved": True, "showOrphans": True, "collapse-color-groups": False, "colorGroups": [{"query": "path:Companies", "color": {"a": 1, "rgb": 3447003}}, {"query": "path:Sectors", "color": {"a": 1, "rgb": 15105570}}, {"query": "path:Investment Profiles", "color": {"a": 1, "rgb": 3066993}}]}, indent=2))

    # Sector and market canvases
    market_nodes = []
    market_edges = []
    for si, (sector, companies) in enumerate(by_sector.items()):
        angle = 2 * math.pi * si / len(by_sector)
        sx, sy = int(1600 * math.cos(angle)), int(1100 * math.sin(angle))
        sector_id = f"sector-{si:02d}"
        market_nodes.append({"id": sector_id, "type": "file", "file": f"Sectors/{clean_file_name(sector)}.md", "x": sx, "y": sy, "width": 340, "height": 220, "color": "4"})
        sector_nodes = [{"id": "sector", "type": "file", "file": f"Sectors/{clean_file_name(sector)}.md", "x": 0, "y": 0, "width": 360, "height": 220, "color": "4"}]
        sector_edges = []
        for ci, company in enumerate(companies):
            a = 2 * math.pi * ci / len(companies)
            x, y = int(650 * math.cos(a)), int(480 * math.sin(a))
            cid = f"company-{si:02d}-{ci:02d}"
            file_path = f"Companies/{clean_file_name(company['name'])}.md"
            sector_nodes.append({"id": cid, "type": "file", "file": file_path, "x": x, "y": y, "width": 280, "height": 170})
            sector_edges.append({"id": f"edge-{ci:02d}", "fromNode": "sector", "toNode": cid})
            mx, my = sx + int(520 * math.cos(a)), sy + int(360 * math.sin(a))
            market_nodes.append({"id": cid, "type": "file", "file": file_path, "x": mx, "y": my, "width": 260, "height": 150})
            market_edges.append({"id": f"market-edge-{si:02d}-{ci:02d}", "fromNode": sector_id, "toNode": cid})
        write_text(ROOT / "Maps" / f"{clean_file_name(sector)} Map.canvas", json.dumps({"nodes": sector_nodes, "edges": sector_edges}, ensure_ascii=False, indent=2))
    write_text(ROOT / "Maps" / "Market Landscape.canvas", json.dumps({"nodes": market_nodes, "edges": market_edges}, ensure_ascii=False, indent=2))


In [ ]:
# 4. Generate exactly 100 deterministic fictional company records — no files yet

# Reset the global RNG so rerunning this cell produces the identical universe.
random.seed(SEED)
records = make_company_records()

assert len(records) == 100
assert len({r["id"] for r in records}) == 100
assert len({r["name"] for r in records}) == 100
assert {r["id"] for r in records} == {f"SYN-{i:03d}" for i in range(1, 101)}

print("Companies generated:", len(records))
print("First ID and company:", records[0]["id"], records[0]["name"])
print("Last ID and company:", records[-1]["id"], records[-1]["name"])


In [ ]:
# 5. Inspect the synthetic universe before creating the vault

def display_table(rows, columns, title=None):
    if title:
        display(Markdown(f"### {title}"))
    head = "".join(f"<th style='padding:6px'>{escape(str(c))}</th>" for c in columns)
    body = "".join("<tr>" + "".join(
        f"<td style='padding:6px;border-top:1px solid #ddd'>{escape(str(row.get(c, '')))}</td>"
        for c in columns) + "</tr>" for row in rows)
    display(HTML(f"<div style='overflow-x:auto'><table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table></div>"))

sector_counts = Counter(r["sector"] for r in records)
profile_counts = Counter(r["investment_style"] for r in records)
stage_counts = Counter(r["stage"] for r in records)

display_table([{"Sector": k, "Companies": v} for k, v in sorted(sector_counts.items())], ["Sector", "Companies"], "Sector distribution")
display_table([{"Profile": k, "Companies": v} for k, v in sorted(profile_counts.items())], ["Profile", "Companies"], "Investment-profile distribution")
display_table([{"Stage": k, "Companies": v} for k, v in sorted(stage_counts.items())], ["Stage", "Companies"], "Lifecycle distribution")


In [ ]:
# 6. Validate every generated record before any file creation

required_fields = set(records[0])
record_checks = []
for r in records:
    reconstructed_ev = round(r["equity_value_usd_m"] + r["debt_usd_m"] - r["cash_usd_m"], 1)
    record_checks.append({
        "id": r["id"],
        "complete_schema": set(r) == required_fields,
        "ev_equation": abs(reconstructed_ev - r["enterprise_value_usd_m"]) <= 0.2,
        "known_sector": r["sector"] in SECTORS,
        "score_ranges": 1 <= r["moat_score"] <= 10 and 1 <= r["management_score"] <= 10 and 1 <= r["execution_risk_score"] <= 10,
    })

validation_summary = {
    "records": len(record_checks),
    "complete_schema": all(x["complete_schema"] for x in record_checks),
    "enterprise_value_equations": all(x["ev_equation"] for x in record_checks),
    "known_sectors": all(x["known_sector"] for x in record_checks),
    "qualitative_score_ranges": all(x["score_ranges"] for x in record_checks),
}
display_table([{"Control": k, "Result": v} for k, v in validation_summary.items()], ["Control", "Result"], "Pre-write validation")
assert all(v is True or k == "records" for k, v in validation_summary.items())


## 7. Build the complete Obsidian vault in temporary staging

The next cell writes only to `STAGING_ROOT`, never to Google Drive. The staging folder is regenerated on reruns so the result remains deterministic and free of stale files.


In [ ]:
# Point the writer at temporary staging, then create the original base vault.
ROOT = STAGING_ROOT
create_vault(records)

staged_files = sorted(p for p in STAGING_ROOT.rglob("*") if p.is_file())
print("Staged vault created:", STAGING_ROOT)
print("Files staged:", len(staged_files))
print("Company notes:", len(list((STAGING_ROOT / "Companies").glob("*.md"))))
print("Canvas maps:", len(list((STAGING_ROOT / "Maps").glob("*.canvas"))))
print("Drive files written: 0")


In [ ]:
# 8. Validate Obsidian links, Canvas JSON and master CSV

markdown_files = list(STAGING_ROOT.rglob("*.md"))
known_rel = {str(p.relative_to(STAGING_ROOT).with_suffix("")) for p in markdown_files}
known_base = {p.stem for p in markdown_files}
link_pattern = re.compile(r"\[\[([^\]|#]+)")
unresolved = []
for path in markdown_files:
    for target in link_pattern.findall(path.read_text(encoding="utf-8")):
        if target not in known_rel and Path(target).name not in known_base:
            if not (STAGING_ROOT / f"{target}.canvas").exists() and not (STAGING_ROOT / target).exists():
                unresolved.append({"source": str(path.relative_to(STAGING_ROOT)), "target": target})

canvas_errors = []
for path in STAGING_ROOT.rglob("*.canvas"):
    try:
        data = json.loads(path.read_text(encoding="utf-8"))
        if not isinstance(data.get("nodes"), list) or not isinstance(data.get("edges"), list):
            canvas_errors.append(str(path.relative_to(STAGING_ROOT)))
    except Exception as exc:
        canvas_errors.append(f"{path.relative_to(STAGING_ROOT)}: {exc}")

with (STAGING_ROOT / "Data/company_master.csv").open(encoding="utf-8", newline="") as f:
    csv_rows = list(csv.DictReader(f))

vault_validation = {
    "company_notes": len(list((STAGING_ROOT / "Companies").glob("*.md"))),
    "master_csv_rows": len(csv_rows),
    "markdown_notes": len(markdown_files),
    "canvas_files": len(list(STAGING_ROOT.rglob("*.canvas"))),
    "unresolved_wikilinks": len(unresolved),
    "canvas_errors": len(canvas_errors),
}
display_table([{"Validation": k, "Result": v} for k, v in vault_validation.items()], ["Validation", "Result"], "Staged-vault validation")
assert vault_validation["company_notes"] == 100
assert vault_validation["master_csv_rows"] == 100
assert not unresolved and not canvas_errors


In [ ]:
# 9. Produce a reproducibility fingerprint and inspect representative files

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest = [{
    "path": str(p.relative_to(STAGING_ROOT)),
    "bytes": p.stat().st_size,
    "sha256": sha256(p),
} for p in staged_files]

manifest_digest = hashlib.sha256("\n".join(f"{x['path']}|{x['bytes']}|{x['sha256']}" for x in manifest).encode()).hexdigest()
print("Full manifest SHA-256:", manifest_digest)
display_table(manifest[:20], ["path", "bytes", "sha256"], "First 20 staged files")

sample = STAGING_ROOT / "Companies" / "Asterion Payments.md"
display(Markdown("### Representative Obsidian company note"))
print("\n".join(sample.read_text(encoding="utf-8").splitlines()[:55]))


In [ ]:
# 10. Create a portable ZIP from staging and verify its contents

WORK_ROOT.mkdir(parents=True, exist_ok=True)
archive_base = str(STAGING_ZIP.with_suffix(""))
created_zip = Path(shutil.make_archive(archive_base, "zip", root_dir=STAGING_ROOT.parent, base_dir=STAGING_ROOT.name))

import zipfile
with zipfile.ZipFile(created_zip) as zf:
    bad_member = zf.testzip()
    zip_members = zf.namelist()

print("ZIP:", created_zip)
print("ZIP bytes:", created_zip.stat().st_size)
print("ZIP members:", len(zip_members))
print("Integrity error:", bad_member)
assert bad_member is None


In [ ]:
# 11. Preview the Drive mutation—still no Drive writes

drive_manifest = [{
    "action": "BLOCKED — target exists" if TARGET_ROOT.exists() else "CREATE DIRECTORY TREE",
    "target": str(TARGET_ROOT),
    "source_files": len(staged_files),
    "source_manifest_sha256": manifest_digest,
}, {
    "action": "CREATE ZIP COPY" if not (MY_DRIVE / created_zip.name).exists() else "BLOCKED — ZIP exists",
    "target": str(MY_DRIVE / created_zip.name),
    "source_files": 1,
    "source_manifest_sha256": sha256(created_zip),
}]
display_table(drive_manifest, ["action", "target", "source_files", "source_manifest_sha256"], "Proposed Drive mutations")
print("COMMIT_TO_DRIVE:", COMMIT_TO_DRIVE)


## 12. Human commit gate

Before committing, verify:

- there are exactly 100 company notes and 100 CSV rows;
- there are no unresolved wikilinks or Canvas errors;
- the distributions and representative company note look sensible;
- the proposed Drive target is correct;
- the target does not already exist.

The commit deliberately refuses to replace anything. This is an initial-creation notebook, not a migration or update tool.


In [ ]:
# Commit a new vault and ZIP to Drive only when explicitly authorized.

committed = []
if not COMMIT_TO_DRIVE:
    print("PREVIEW COMPLETE — the vault exists only in temporary staging.")
    print("No files were copied into Google Drive.")
else:
    if TARGET_ROOT.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing vault: {TARGET_ROOT}. "
            "Choose a different TARGET_VAULT_NAME for a parallel rebuild."
        )
    drive_zip = MY_DRIVE / created_zip.name
    if drive_zip.exists():
        raise FileExistsError(f"Refusing to overwrite existing ZIP: {drive_zip}")

    # Build under a temporary sibling and publish by rename only after copy succeeds.
    publish_temp = MY_DRIVE / f".{TARGET_VAULT_NAME}.publishing"
    if publish_temp.exists():
        raise FileExistsError(f"Stale publishing directory requires human review: {publish_temp}")
    shutil.copytree(STAGING_ROOT, publish_temp)
    publish_temp.rename(TARGET_ROOT)
    shutil.copy2(created_zip, drive_zip)
    committed = [str(TARGET_ROOT), str(drive_zip)]
    print("Committed:", committed)


In [ ]:
# 13. Final execution summary and Day-1 handoff

final_summary = {
    "purpose": "Initial Day-0 creation of the original synthetic Obsidian vault",
    "seed": SEED,
    "synthetic_companies": len(records),
    "sectors": len(sector_counts),
    "investment_profiles": dict(sorted(profile_counts.items())),
    "company_notes": vault_validation["company_notes"],
    "markdown_notes": vault_validation["markdown_notes"],
    "canvas_maps": vault_validation["canvas_files"],
    "unresolved_wikilinks": vault_validation["unresolved_wikilinks"],
    "manifest_sha256": manifest_digest,
    "drive_commit_authorized": COMMIT_TO_DRIVE,
    "drive_paths_created": committed,
    "excluded_from_day0": ["SYN-101", "SYN-102", "SCN-001", "SCN-002", "opportunity scoring"],
    "next_step": "Open 00 Home.md in Obsidian and inspect the company screener and maps.",
}
print(json.dumps(final_summary, indent=2, ensure_ascii=False))


## What this notebook gives you

The entire Day-0 vault is reproducible from one file. The random process is seeded; every generation rule is visible; the Obsidian writer is included; validation happens before Drive publication; and the output remains plain Markdown, CSV, JSON Canvas and Obsidian configuration files.

Later notebooks should consume this vault as prior state. They should not be mixed into the Day-0 generator, because preserving the boundary between initial data creation and subsequent analytical loops is part of the audit trail.
